# Поиск аномалий

Методы обнаружения аномалий, как следует из названия, позволяют находить необычные объекты в выборке. Но что такое "необычные" и совпадает ли это определение у разных методов?

Начнём с поиска аномалий в текстах: научимся отличать вопросы о программировании от текстов из 20newsgroups про религию.

Подготовьте данные: в обучающую выборку возьмите 20 тысяч текстов из датасета Stack Overflow, а тестовую выборку сформируйте из 10 тысяч текстов со Stack Overflow и 100 текстов из класса soc.religion.christian датасета 20newsgroups (очень пригодится функция `fetch_20newsgroups(categories=['soc.religion.christian'])`). Тексты про программирование будем считать обычными, а тексты про религию — аномальными.

In [ ]:
#code here
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Загрузка Stack Overflow данных (используем code_search_net)
print("Загрузка Stack Overflow данных...")
so_dataset = load_dataset('code_search_net', 'python', split='train', trust_remote_code=True)
so_texts = [item['func_documentation_string'] for item in so_dataset.select(range(30000)) if item['func_documentation_string'] is not None]
so_texts = so_texts[:30000]
print(f"Загружено {len(so_texts)} текстов Stack Overflow")

# Разделение на обучение (20k) и тест (10k)
train_texts_so = so_texts[:20000]
test_texts_so = so_texts[20000:30000]

# Загрузка религиозных текстов
print("Загрузка 20newsgroups (soc.religion.christian)...")
religion = fetch_20newsgroups(categories=['soc.religion.christian'], shuffle=True, random_state=42, remove=('headers', 'footers', 'quotes'))
religion_texts = religion.data[:100]
print(f"Загружено {len(religion_texts)} религиозных текстов")

# Формирование тестовой выборки и меток
test_texts = test_texts_so + religion_texts
test_labels = [0] * len(test_texts_so) + [1] * len(religion_texts)  # 0 – норма, 1 – аномалия

**(1 балл)**

Проверьте качество выделения аномалий (precision и recall на тестовой выборке, если считать аномалии положительным классов, а обычные тексты — отрицательным) для IsolationForest. В качестве признаков используйте TF-IDF, где словарь и IDF строятся по обучающей выборке. Не забудьте подобрать гиперпараметры.

In [ ]:
#code here
# TF-IDF векторизация
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = vectorizer.fit_transform(train_texts_so)
X_test = vectorizer.transform(test_texts)

# Подбор contamination и n_estimators
contamination_values = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
n_estimators_values = [50, 100, 200]
best_precision = 0
best_recall = 0
best_params = {}

for contamination in contamination_values:
    for n_estimators in n_estimators_values:
        iso_forest = IsolationForest(contamination=contamination, n_estimators=n_estimators, random_state=42)
        iso_forest.fit(X_train)
        preds = iso_forest.predict(X_test)
        preds_binary = [1 if p == -1 else 0 for p in preds]
        precision = precision_score(test_labels, preds_binary, zero_division=0)
        recall = recall_score(test_labels, preds_binary)
        print(f"contamination={contamination}, n_estimators={n_estimators}: precision={precision:.4f}, recall={recall:.4f}")
        if recall > best_recall:
            best_recall = recall
            best_precision = precision
            best_params = {'contamination': contamination, 'n_estimators': n_estimators}

print(f"\nЛучшие параметры: {best_params}, precision={best_precision:.4f}, recall={best_recall:.4f}")

**(5 баллов)**

Скорее всего, качество оказалось не на высоте. Разберитесь, в чём дело:
* посмотрите на тексты, которые выделяются как аномальные, а также на слова, соответствующие их ненулевым признакам
* изучите признаки аномальных текстов
* посмотрите на тексты из обучающей выборки, ближайшие к аномальным; действительно ли они похожи по признакам?

Сделайте выводы и придумайте, как избавиться от этих проблем. Предложите варианты двух типов: (1) в рамках этих же признаков (но которые, возможно, будут считаться по другим наборам данных) и методов и (2) без ограничений на изменения. Реализуйте эти варианты и проверьте их качество.

In [ ]:
#code here
# Используем лучшие параметры для анализа
iso_forest_best = IsolationForest(contamination=best_params['contamination'], n_estimators=best_params['n_estimators'], random_state=42)
iso_forest_best.fit(X_train)
preds = iso_forest_best.predict(X_test)
preds_binary = [1 if p == -1 else 0 for p in preds]

# Вывод аномальных текстов
anomaly_indices = [i for i, p in enumerate(preds_binary) if p == 1]
print(f"Найдено аномалий: {len(anomaly_indices)}")
print("Примеры аномальных текстов:")
for idx in anomaly_indices[:5]:
    print(f"--- Текст {idx} (истинная метка: {test_labels[idx]}) ---")
    print(test_texts[idx][:500])
    print()

# Топ-слов для аномалий
feature_names = vectorizer.get_feature_names_out()
for idx in anomaly_indices[:5]:
    print(f"Топ-10 слов для аномального текста {idx}:")
    row = X_test[idx].toarray().flatten()
    top_indices = row.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices if row[i] > 0]
    print(top_words)
    print()

# Ближайшие обычные тексты из обучающей выборки
from sklearn.metrics.pairwise import cosine_similarity
if anomaly_indices:
    anom_vec = X_test[anomaly_indices[0]]
    sims = cosine_similarity(anom_vec, X_train).flatten()
    top_sim_indices = sims.argsort()[-5:][::-1]
    print("Ближайшие обучающие тексты (обычные) к первой аномалии:")
    for i, idx_train in enumerate(top_sim_indices):
        print(f"Близость {sims[idx_train]:.4f}: {train_texts_so[idx_train][:200]}...")
        print()

# Улучшение (1) – расширение обучающей выборки другими категориями 20newsgroups
print("=== Улучшение (1): расширение обучающей выборки ===")
other_cats = ['comp.graphics', 'rec.sport.baseball', 'sci.space']
other_news = fetch_20newsgroups(categories=other_cats, remove=('headers', 'footers', 'quotes'), shuffle=True, random_state=42)
other_texts = other_news.data[:5000]
train_texts_expanded = train_texts_so + other_texts
print(f"Размер расширенной обучающей выборки: {len(train_texts_expanded)}")

vectorizer_exp = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_exp = vectorizer_exp.fit_transform(train_texts_expanded)
X_test_exp = vectorizer_exp.transform(test_texts)

iso_exp = IsolationForest(contamination=best_params['contamination'], n_estimators=best_params['n_estimators'], random_state=42)
iso_exp.fit(X_train_exp)
preds_exp = iso_exp.predict(X_test_exp)
preds_binary_exp = [1 if p == -1 else 0 for p in preds_exp]
precision_exp = precision_score(test_labels, preds_binary_exp, zero_division=0)
recall_exp = recall_score(test_labels, preds_binary_exp)
print(f"Улучшение (1): precision={precision_exp:.4f}, recall={recall_exp:.4f}")

# Улучшение (2) – использование эмбеддингов Sentence-BERT
print("\n=== Улучшение (2): эмбеддинги Sentence-BERT ===")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
train_sample = train_texts_so[:5000]  # для скорости
print("Вычисление эмбеддингов для обучающей выборки...")
train_emb = model.encode(train_sample, show_progress_bar=True)
print("Вычисление эмбеддингов для тестовой выборки...")
test_emb = model.encode(test_texts, show_progress_bar=True)

iso_emb = IsolationForest(contamination=best_params['contamination'], n_estimators=best_params['n_estimators'], random_state=42)
iso_emb.fit(train_emb)
preds_emb = iso_emb.predict(test_emb)
preds_binary_emb = [1 if p == -1 else 0 for p in preds_emb]
precision_emb = precision_score(test_labels, preds_binary_emb, zero_division=0)
recall_emb = recall_score(test_labels, preds_binary_emb)
print(f"Улучшение (2): precision={precision_emb:.4f}, recall={recall_emb:.4f}")

### Эксперимент только с изменением датасета

In [ ]:
#code here
# Повтор улучшения (1) – расширение датасета
from sklearn.datasets import fetch_20newsgroups

other_cats = ['comp.graphics', 'rec.sport.baseball', 'sci.space']
other_news = fetch_20newsgroups(categories=other_cats, remove=('headers', 'footers', 'quotes'), shuffle=True, random_state=42)
other_texts = other_news.data[:5000]
train_texts_expanded = train_texts_so + other_texts
print(f"Размер расширенной обучающей выборки: {len(train_texts_expanded)}")

vectorizer_exp = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_exp = vectorizer_exp.fit_transform(train_texts_expanded)
X_test_exp = vectorizer_exp.transform(test_texts)

iso_exp = IsolationForest(contamination=0.01, n_estimators=100, random_state=42)  # фиксированные параметры
iso_exp.fit(X_train_exp)
preds_exp = iso_exp.predict(X_test_exp)
preds_binary_exp = [1 if p == -1 else 0 for p in preds_exp]
precision_exp = precision_score(test_labels, preds_binary_exp, zero_division=0)
recall_exp = recall_score(test_labels, preds_binary_exp)
print(f"Эксперимент (только изменение датасета): precision={precision_exp:.4f}, recall={recall_exp:.4f}")

### Эксперимент с любыми изменениями

In [ ]:
#code here
# Эксперимент с эмбеддингами Sentence-BERT
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
train_sample = train_texts_so[:5000]
print("Вычисление эмбеддингов для обучающей выборки...")
train_emb = model.encode(train_sample, show_progress_bar=True)
print("Вычисление эмбеддингов для тестовой выборки...")
test_emb = model.encode(test_texts, show_progress_bar=True)

iso_emb = IsolationForest(contamination=0.01, n_estimators=100, random_state=42)
iso_emb.fit(train_emb)
preds_emb = iso_emb.predict(test_emb)
preds_binary_emb = [1 if p == -1 else 0 for p in preds_emb]
precision_emb = precision_score(test_labels, preds_binary_emb, zero_division=0)
recall_emb = recall_score(test_labels, preds_binary_emb)
print(f"Эксперимент (любые изменения - эмбеддинги): precision={precision_emb:.4f}, recall={recall_emb:.4f}")

Подготовьте выборку: удалите столбцы `['id', 'date', 'price', 'zipcode']`, сформируйте обучающую и тестовую выборки по 10 тысяч домов.

Добавьте в тестовую выборку 10 новых объектов, в каждом из которых испорчен ровно один признак — например, это может быть дом из другого полушария, из далёкого прошлого или будущего, с площадью в целый штат или с таким числом этажей, что самолётам неплохо бы его облетать стороной.

Посмотрим на методы обнаружения аномалий на более простых данных — уж на табличном датасете с 19 признаками всё должно работать как надо!

Скачайте данные о стоимости домов: https://www.kaggle.com/harlfoxem/housesalesprediction/data

In [ ]:
#code here
import pandas as pd
import numpy as np

# Загрузка данных (файл должен быть в рабочей директории)
df = pd.read_csv('kc_house_data.csv')
print(f"Исходный размер: {df.shape}")

# Удаление ненужных столбцов
df.drop(columns=['id', 'date', 'price', 'zipcode'], inplace=True, errors='ignore')
print(f"Размер после удаления: {df.shape}")

# Перемешивание и разделение на обучающую и тестовую выборки (по 10 тыс.)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = df.iloc[:10000].copy()
test_df = df.iloc[10000:20000].copy()
print(f"Обучающая выборка: {train_df.shape}, тестовая: {test_df.shape}")

# Создание 10 аномальных объектов
anomalies = []
# Аномалия 1: огромная жилая площадь
anom1 = train_df.iloc[0].copy()
anom1['sqft_living'] = 1e6
anomalies.append(anom1)

# Аномалия 2: очень много этажей
anom2 = train_df.iloc[1].copy()
anom2['floors'] = 100
anomalies.append(anom2)

# Аномалия 3: слишком старый дом
anom3 = train_df.iloc[2].copy()
anom3['yr_built'] = 1600
anomalies.append(anom3)

# Аномалия 4: слишком маленькая жилая площадь
anom4 = train_df.iloc[3].copy()
anom4['sqft_living'] = 1
anomalies.append(anom4)

# Аномалия 5: огромный участок
anom5 = train_df.iloc[4].copy()
anom5['sqft_lot'] = 1e8
anomalies.append(anom5)

# Аномалия 6: отрицательное число ванных
anom6 = train_df.iloc[5].copy()
anom6['bathrooms'] = -10
anomalies.append(anom6)

# Аномалия 7: слишком много спален
anom7 = train_df.iloc[6].copy()
anom7['bedrooms'] = 50
anomalies.append(anom7)

# Аномалия 8: очень высокий этаж
anom8 = train_df.iloc[7].copy()
anom8['floors'] = 500
anomalies.append(anom8)

# Аномалия 9: отрицательный год ремонта
anom9 = train_df.iloc[8].copy()
anom9['yr_renovated'] = -1000
anomalies.append(anom9)

# Аномалия 10: некорректное значение waterfront
anom10 = train_df.iloc[9].copy()
anom10['waterfront'] = 10
anomalies.append(anom10)

# Добавление аномалий к тестовой выборке
anom_df = pd.DataFrame(anomalies)
test_df = pd.concat([test_df, anom_df], ignore_index=True)

# Метки: 0 – норма, 1 – аномалия
test_labels_houses = [0] * (10000) + [1] * 10
print(f"Тестовая выборка теперь размером {test_df.shape}, из них аномалий: {sum(test_labels_houses)}")

**Задание 9. (2 балла)**

Примените IsolationForest для поиска аномалий в этих данных, запишите их качество (как и раньше, это precision и recall). Проведите исследование:

Нарисуйте распределения всех признаков и обозначьте на этих распределениях объекты, которые признаны аномальными.

In [ ]:
#code here
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score
import matplotlib.pyplot as plt

# Масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df)
X_test = scaler.transform(test_df)

# Обучение IsolationForest
iso_house = IsolationForest(contamination=0.01, random_state=42)
iso_house.fit(X_train)
preds_house = iso_house.predict(X_test)
preds_binary_house = [1 if p == -1 else 0 for p in preds_house]

precision_house = precision_score(test_labels_houses, preds_binary_house, zero_division=0)
recall_house = recall_score(test_labels_houses, preds_binary_house)
print(f"Precision: {precision_house:.4f}, Recall: {recall_house:.4f}")

# Индексы предсказанных аномалий
anomaly_indices_house = [i for i, p in enumerate(preds_binary_house) if p == 1]

# Построение распределений признаков с отметками аномалий
features = test_df.columns
n_features = len(features)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten()

for i, feature in enumerate(features):
    ax = axes[i]
    ax.hist(test_df[feature], bins=50, alpha=0.5, label='Все объекты')
    for idx in anomaly_indices_house:
        if idx < len(test_df):
            ax.axvline(test_df.loc[idx, feature], color='red', linestyle='--', alpha=0.7)
    ax.set_title(feature)
    ax.legend()

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()